<a href="https://colab.research.google.com/github/mortadarammal/LLM-as-a-Judge/blob/iyad/RAG1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS50 Project — LLM-as-a-Judge for RAG

## Objective
Build a Retrieval-Augmented Generation (RAG) pipeline, test multiple open-source LLMs, and evaluate them using an LLM-as-a-Judge approach.

## First milestone
Set up the environment, prepare the dataset, build the vector store, and run a first end-to-end RAG test.

In [9]:
!pip install -q \
  transformers \
  accelerate \
  bitsandbytes \
  sentence-transformers \
  faiss-cpu \
  datasets \
  pandas \
  matplotlib \
  tqdm \
  langchain \
  langchain-core \
  langchain-community \
  langchain-huggingface \
  langchain-text-splitters \
  ragas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.2/178.2 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.7/360.7 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 k

In [10]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected")

CUDA available: True
GPU name: Tesla T4


In [11]:
from huggingface_hub import login
from getpass import getpass

hf_token = getpass("Enter your Hugging Face token: ")
login(hf_token)

Enter your Hugging Face token: ··········


In [13]:
import os
import gc
import time
import json
import pandas as pd
from tqdm import tqdm

import torch
from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline,
    BitsAndBytesConfig
)

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import PromptTemplate

In [62]:
CONFIG = {
    "dataset_name": "pubmed_qa",
    "dataset_subset": "pqa_labeled",
    "num_docs": 200,
    "num_questions": 10,
    "chunk_size": 512,
    "chunk_overlap": 64,
    "top_k": 3,
    "max_new_tokens": 128,
    "temperature": 0.0,
    "do_sample": False,
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "generator_models": {
        "qwen_1_5b": "Qwen/Qwen2.5-1.5B-Instruct",
        "mistral_7b": "mistralai/Mistral-7B-Instruct-v0.3"
    },
    "results_dir": "results"
}

In [63]:
def run_rag_experiment_for_model(model_key, model_name, train_data, retriever, prompt_template, num_questions):
    print(f"\nRunning experiment for model: {model_key} -> {model_name}")

    tokenizer, model = load_model_4bit(model_name)

    text_gen = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=CONFIG["max_new_tokens"],
        temperature=CONFIG["temperature"],
        do_sample=CONFIG["do_sample"],
        return_full_text=False,
        pad_token_id=tokenizer.eos_token_id
    )

    model_results = []

    for i in tqdm(range(num_questions), desc=f"Running {model_key}"):
        question = train_data[i]["question"]
        ground_truth = train_data[i]["final_decision"]

        start_time = time.time()
        rag_output = ask_rag(
            question=question,
            retriever=retriever,
            text_gen_pipeline=text_gen,
            prompt_template=prompt_template
        )
        end_time = time.time()

        retrieved_contexts = [doc.page_content for doc in rag_output["source_documents"]]
        retrieved_metadata = [doc.metadata for doc in rag_output["source_documents"]]

        model_results.append({
            "model_key": model_key,
            "model_name": model_name,
            "question_id": i,
            "question": question,
            "ground_truth": ground_truth,
            "generated_answer": rag_output["result"],
            "retrieved_contexts": retrieved_contexts,
            "retrieved_metadata": retrieved_metadata,
            "inference_time_sec": round(end_time - start_time, 4)
        })

    results_df = pd.DataFrame(model_results)

    save_path = os.path.join(CONFIG["results_dir"], f"{model_key}_results.csv")
    results_df.to_csv(save_path, index=False)

    avg_time = results_df["inference_time_sec"].mean()

    print(f"\nFinished model: {model_key}")
    print(f"Results saved to: {save_path}")
    print(f"Average inference time: {avg_time:.2f} seconds")

    del model
    del tokenizer
    del text_gen
    clear_gpu_memory()

    return results_df

In [64]:
qwen_results_df = run_rag_experiment_for_model(
    model_key="qwen_1_5b",
    model_name=CONFIG["generator_models"]["qwen_1_5b"],
    train_data=train_data,
    retriever=retriever,
    prompt_template=rag_prompt,
    num_questions=CONFIG["num_questions"]
)


Running experiment for model: qwen_1_5b -> Qwen/Qwen2.5-1.5B-Instruct


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Running qwen_1_5b: 100%|██████████| 10/10 [01:27<00:00,  8.73s/it]



Finished model: qwen_1_5b
Results saved to: results/qwen_1_5b_results.csv
Average inference time: 8.73 seconds


In [65]:
mistral_results_df = run_rag_experiment_for_model(
    model_key="mistral_7b",
    model_name=CONFIG["generator_models"]["mistral_7b"],
    train_data=train_data,
    retriever=retriever,
    prompt_template=rag_prompt,
    num_questions=CONFIG["num_questions"]
)


Running experiment for model: mistral_7b -> mistralai/Mistral-7B-Instruct-v0.3


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Running mistral_7b: 100%|██████████| 10/10 [01:31<00:00,  9.15s/it]



Finished model: mistral_7b
Results saved to: results/mistral_7b_results.csv
Average inference time: 9.14 seconds


In [69]:
all_results_df = pd.concat([qwen_results_df, mistral_results_df], ignore_index=True)

print("Combined results shape:", all_results_df.shape)
all_results_df.head(50)

Combined results shape: (20, 9)


,model_key,model_name,question_id,question,ground_truth,generated_answer,retrieved_contexts,retrieved_metadata,inference_time_sec
0,qwen_1_5b,Qwen/Qwen2.5-1.5B-Instruct,0,Do mitochondria play a role in remodelling lac...,yes,"Yes, according to the provided information, mi...",[Programmed cell death (PCD) is the regulated ...,"[{'source_id': 0, 'chunk_id': 0, 'question': '...",7.5915
1,qwen_1_5b,Qwen/Qwen2.5-1.5B-Instruct,1,Landolt C and snellen e acuity: differences in...,no,The mean difference between Landolt C acuity (...,[Differences between Landolt C acuity (LR) and...,"[{'source_id': 1, 'chunk_id': 2, 'question': '...",6.2225
2,qwen_1_5b,Qwen/Qwen2.5-1.5B-Instruct,2,"Syncope during bathing in infants, a pediatric...",yes,The provided context does not support the spec...,[Eight infants aged 2 to 15 months were admitt...,"[{'source_id': 2, 'chunk_id': 1, 'question': '...",10.6633
3,qwen_1_5b,Qwen/Qwen2.5-1.5B-Instruct,3,Are the long-term results of the transanal pul...,no,"No, the long-term outcomes of the transanal pu...",[The transanal endorectal pull-through (TERPT)...,"[{'source_id': 3, 'chunk_id': 0, 'question': '...",10.4322
4,qwen_1_5b,Qwen/Qwen2.5-1.5B-Instruct,4,Can tailored interventions increase mammograph...,yes,"Yes, tailored interventions can increase mammo...","[Compared to usual care alone, telephone couns...","[{'source_id': 4, 'chunk_id': 5, 'question': '...",5.1358
5,qwen_1_5b,Qwen/Qwen2.5-1.5B-Instruct,5,Double balloon enteroscopy: is it efficacious ...,yes,"No, the provided information does not support ...",[Sixty consecutive TAVI patients were randomiz...,"[{'source_id': 59, 'chunk_id': 1, 'question': ...",12.8265
6,qwen_1_5b,Qwen/Qwen2.5-1.5B-Instruct,6,30-Day and 1-year mortality in emergency gener...,maybe,"Yes, 30-day and 1-year mortality in emergency ...",[This was a retrospective study of patients wh...,"[{'source_id': 6, 'chunk_id': 1, 'question': '...",14.4667
7,qwen_1_5b,Qwen/Qwen2.5-1.5B-Instruct,7,Is adjustment for reporting heterogeneity nece...,no,"Yes, adjustment for reporting heterogeneity is...",[Anchoring vignettes are brief texts describin...,"[{'source_id': 7, 'chunk_id': 0, 'question': '...",4.0381
8,qwen_1_5b,Qwen/Qwen2.5-1.5B-Instruct,8,Do mutations causing low HDL-C promote increas...,no,"Yes, mutations causing low HDL-C can promote i...",[Carotid intima-media thickness (cIMT) measure...,"[{'source_id': 8, 'chunk_id': 1, 'question': '...",10.7967
9,qwen_1_5b,Qwen/Qwen2.5-1.5B-Instruct,9,A short stay or 23-hour ward in a general and ...,yes,"Yes, the short stay ward was found to be effec...",[We evaluated the usefulness of a short stay o...,"[{'source_id': 9, 'chunk_id': 0, 'question': '...",5.1450


In [67]:
time_comparison = all_results_df.groupby("model_key")["inference_time_sec"].mean().reset_index()
time_comparison = time_comparison.sort_values("inference_time_sec")

print(time_comparison)

    model_key  inference_time_sec
1   qwen_1_5b             8.73183
0  mistral_7b             9.14437


In [71]:
print(all_results_df.columns.tolist())
all_results_df.head(50)

['model_key', 'model_name', 'question_id', 'question', 'ground_truth', 'generated_answer', 'retrieved_contexts', 'retrieved_metadata', 'inference_time_sec']


,model_key,model_name,question_id,question,ground_truth,generated_answer,retrieved_contexts,retrieved_metadata,inference_time_sec
0,qwen_1_5b,Qwen/Qwen2.5-1.5B-Instruct,0,Do mitochondria play a role in remodelling lac...,yes,"Yes, according to the provided information, mi...",[Programmed cell death (PCD) is the regulated ...,"[{'source_id': 0, 'chunk_id': 0, 'question': '...",7.5915
1,qwen_1_5b,Qwen/Qwen2.5-1.5B-Instruct,1,Landolt C and snellen e acuity: differences in...,no,The mean difference between Landolt C acuity (...,[Differences between Landolt C acuity (LR) and...,"[{'source_id': 1, 'chunk_id': 2, 'question': '...",6.2225
2,qwen_1_5b,Qwen/Qwen2.5-1.5B-Instruct,2,"Syncope during bathing in infants, a pediatric...",yes,The provided context does not support the spec...,[Eight infants aged 2 to 15 months were admitt...,"[{'source_id': 2, 'chunk_id': 1, 'question': '...",10.6633
3,qwen_1_5b,Qwen/Qwen2.5-1.5B-Instruct,3,Are the long-term results of the transanal pul...,no,"No, the long-term outcomes of the transanal pu...",[The transanal endorectal pull-through (TERPT)...,"[{'source_id': 3, 'chunk_id': 0, 'question': '...",10.4322
4,qwen_1_5b,Qwen/Qwen2.5-1.5B-Instruct,4,Can tailored interventions increase mammograph...,yes,"Yes, tailored interventions can increase mammo...","[Compared to usual care alone, telephone couns...","[{'source_id': 4, 'chunk_id': 5, 'question': '...",5.1358
5,qwen_1_5b,Qwen/Qwen2.5-1.5B-Instruct,5,Double balloon enteroscopy: is it efficacious ...,yes,"No, the provided information does not support ...",[Sixty consecutive TAVI patients were randomiz...,"[{'source_id': 59, 'chunk_id': 1, 'question': ...",12.8265
6,qwen_1_5b,Qwen/Qwen2.5-1.5B-Instruct,6,30-Day and 1-year mortality in emergency gener...,maybe,"Yes, 30-day and 1-year mortality in emergency ...",[This was a retrospective study of patients wh...,"[{'source_id': 6, 'chunk_id': 1, 'question': '...",14.4667
7,qwen_1_5b,Qwen/Qwen2.5-1.5B-Instruct,7,Is adjustment for reporting heterogeneity nece...,no,"Yes, adjustment for reporting heterogeneity is...",[Anchoring vignettes are brief texts describin...,"[{'source_id': 7, 'chunk_id': 0, 'question': '...",4.0381
8,qwen_1_5b,Qwen/Qwen2.5-1.5B-Instruct,8,Do mutations causing low HDL-C promote increas...,no,"Yes, mutations causing low HDL-C can promote i...",[Carotid intima-media thickness (cIMT) measure...,"[{'source_id': 8, 'chunk_id': 1, 'question': '...",10.7967
9,qwen_1_5b,Qwen/Qwen2.5-1.5B-Instruct,9,A short stay or 23-hour ward in a general and ...,yes,"Yes, the short stay ward was found to be effec...",[We evaluated the usefulness of a short stay o...,"[{'source_id': 9, 'chunk_id': 0, 'question': '...",5.1450


In [73]:
eval_df = all_results_df.copy()

eval_df = eval_df.rename(columns={
    "question": "user_input",
    "generated_answer": "response",
    "ground_truth": "reference",
    "retrieved_contexts": "retrieved_contexts"
})

eval_df = eval_df[[
    "model_key",
    "model_name",
    "question_id",
    "user_input",
    "response",
    "reference",
    "retrieved_contexts"
]]

print("Evaluation DataFrame shape:", eval_df.shape)
eval_df.head(50)

Evaluation DataFrame shape: (20, 7)


,model_key,model_name,question_id,user_input,response,reference,retrieved_contexts
0,qwen_1_5b,Qwen/Qwen2.5-1.5B-Instruct,0,Do mitochondria play a role in remodelling lac...,"Yes, according to the provided information, mi...",yes,[Programmed cell death (PCD) is the regulated ...
1,qwen_1_5b,Qwen/Qwen2.5-1.5B-Instruct,1,Landolt C and snellen e acuity: differences in...,The mean difference between Landolt C acuity (...,no,[Differences between Landolt C acuity (LR) and...
2,qwen_1_5b,Qwen/Qwen2.5-1.5B-Instruct,2,"Syncope during bathing in infants, a pediatric...",The provided context does not support the spec...,yes,[Eight infants aged 2 to 15 months were admitt...
3,qwen_1_5b,Qwen/Qwen2.5-1.5B-Instruct,3,Are the long-term results of the transanal pul...,"No, the long-term outcomes of the transanal pu...",no,[The transanal endorectal pull-through (TERPT)...
4,qwen_1_5b,Qwen/Qwen2.5-1.5B-Instruct,4,Can tailored interventions increase mammograph...,"Yes, tailored interventions can increase mammo...",yes,"[Compared to usual care alone, telephone couns..."
5,qwen_1_5b,Qwen/Qwen2.5-1.5B-Instruct,5,Double balloon enteroscopy: is it efficacious ...,"No, the provided information does not support ...",yes,[Sixty consecutive TAVI patients were randomiz...
6,qwen_1_5b,Qwen/Qwen2.5-1.5B-Instruct,6,30-Day and 1-year mortality in emergency gener...,"Yes, 30-day and 1-year mortality in emergency ...",maybe,[This was a retrospective study of patients wh...
7,qwen_1_5b,Qwen/Qwen2.5-1.5B-Instruct,7,Is adjustment for reporting heterogeneity nece...,"Yes, adjustment for reporting heterogeneity is...",no,[Anchoring vignettes are brief texts describin...
8,qwen_1_5b,Qwen/Qwen2.5-1.5B-Instruct,8,Do mutations causing low HDL-C promote increas...,"Yes, mutations causing low HDL-C can promote i...",no,[Carotid intima-media thickness (cIMT) measure...
9,qwen_1_5b,Qwen/Qwen2.5-1.5B-Instruct,9,A short stay or 23-hour ward in a general and ...,"Yes, the short stay ward was found to be effec...",yes,[We evaluated the usefulness of a short stay o...


In [74]:
print(type(eval_df.loc[0, "retrieved_contexts"]))
print(eval_df.loc[0, "retrieved_contexts"])

<class 'list'>
['Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less', 'PCD was indirectly examined via in vivo cyclosporine A (CsA) treatment. This treatment resulted in lace plant leaves with a significantly lower number of perforations compared to controls, and that displayed mitochondrial dynamics similar to that of non-PCD cells.', 'The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was di

In [75]:
import ast

def ensure_list(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, list):
                return parsed
            return [x]
        except:
            return [x]
    return [str(x)]

eval_df["retrieved_contexts"] = eval_df["retrieved_contexts"].apply(ensure_list)

print(type(eval_df.loc[0, "retrieved_contexts"]))
print(eval_df.loc[0, "retrieved_contexts"][:1])

<class 'list'>
['Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less']


In [76]:
eval_sample_df = eval_df.groupby("model_key").head(3).reset_index(drop=True)

print("Sample shape:", eval_sample_df.shape)
eval_sample_df[["model_key", "user_input", "reference"]]

Sample shape: (6, 7)


,model_key,user_input,reference
0,qwen_1_5b,Do mitochondria play a role in remodelling lac...,yes
1,qwen_1_5b,Landolt C and snellen e acuity: differences in...,no
2,qwen_1_5b,"Syncope during bathing in infants, a pediatric...",yes
3,mistral_7b,Do mitochondria play a role in remodelling lac...,yes
4,mistral_7b,Landolt C and snellen e acuity: differences in...,no
5,mistral_7b,"Syncope during bathing in infants, a pediatric...",yes


In [77]:
def extract_label(text):
    text_lower = str(text).lower().strip()

    if text_lower.startswith("yes"):
        return "yes"
    elif text_lower.startswith("no"):
        return "no"
    elif text_lower.startswith("maybe"):
        return "maybe"
    else:
        if " yes" in text_lower:
            return "yes"
        elif " no" in text_lower:
            return "no"
        elif "maybe" in text_lower:
            return "maybe"
        else:
            return "unknown"

eval_df["predicted_label"] = eval_df["response"].apply(extract_label)

eval_df[["model_key", "reference", "predicted_label", "response"]].head(10)

,model_key,reference,predicted_label,response
0,qwen_1_5b,yes,yes,"Yes, according to the provided information, mi..."
1,qwen_1_5b,no,unknown,The mean difference between Landolt C acuity (...
2,qwen_1_5b,yes,no,The provided context does not support the spec...
3,qwen_1_5b,no,no,"No, the long-term outcomes of the transanal pu..."
4,qwen_1_5b,yes,yes,"Yes, tailored interventions can increase mammo..."
5,qwen_1_5b,yes,no,"No, the provided information does not support ..."
6,qwen_1_5b,maybe,yes,"Yes, 30-day and 1-year mortality in emergency ..."
7,qwen_1_5b,no,yes,"Yes, adjustment for reporting heterogeneity is..."
8,qwen_1_5b,no,yes,"Yes, mutations causing low HDL-C can promote i..."
9,qwen_1_5b,yes,yes,"Yes, the short stay ward was found to be effec..."


In [78]:
eval_df["is_correct_label"] = eval_df["predicted_label"] == eval_df["reference"]

label_accuracy = (
    eval_df.groupby("model_key")["is_correct_label"]
    .mean()
    .reset_index()
    .rename(columns={"is_correct_label": "label_accuracy"})
)

print(label_accuracy)

    model_key  label_accuracy
0  mistral_7b             0.6
1   qwen_1_5b             0.4


In [79]:
unknown_counts = (
    eval_df.groupby("model_key")["predicted_label"]
    .apply(lambda x: (x == "unknown").sum())
    .reset_index(name="unknown_count")
)

print(unknown_counts)

    model_key  unknown_count
0  mistral_7b              1
1   qwen_1_5b              1


In [80]:
summary_df = pd.merge(time_comparison, label_accuracy, on="model_key", how="left")
summary_df = pd.merge(summary_df, unknown_counts, on="model_key", how="left")

summary_df = summary_df.sort_values(by=["label_accuracy", "inference_time_sec"], ascending=[False, True])

summary_df

,model_key,inference_time_sec,label_accuracy,unknown_count
1,mistral_7b,9.14437,0.6,1
0,qwen_1_5b,8.73183,0.4,1
